In [108]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from glob import glob
from pathlib import Path
import sys, io, contextlib

In [109]:
# Get csv files in directory
fn = glob('*.csv')
print(list(fn)) # Print list of csv files

['ADRC_Good_Tune.bbl.csv']


In [110]:
# Define axis names
AXIS_NAMES = ['roll', 'pitch', 'yaw'] # Names for axes, corresponds to 0, 1, 2

def load_blackbox_csv(path):
    """
    Loads a betaflight blackbox CSV, finds length of fileheader (seems to be variabble) and skips before loading as a df
    Handles both setpoint[i] and rcCommand[i] if setpoint isn't logged.

    Input path to csv
    Returns a dataframe containing time, timesteps, setpoints for each axis, and gyro reading for each axis
    Times are in seconds, gyro readings are filtered outptu in blackbox
    """
    # Stuff to read in dataframe consistently because header is a variable length
    with open(path, 'r', newline='') as f:
        lines = f.readlines()

    # Find the line that looks like the actual column header row.
    # In Betaflight logs this row typically starts with "loopIteration"
    header_idx = None
    for i, line in enumerate(lines):
        if line.startswith('"loopIteration"') or line.startswith('loopIteration'):
            header_idx = i
            break

    if header_idx is None:
        raise ValueError(f"Could not find column header row in {path}")
        
    df = pd.read_csv(path, skiprows = header_idx) # Use located start of header
    df.columns = [c.strip().strip("'\"") for c in df.columns]

    time_col = 'time' # Column of timesteps in microseconds
    t = df[time_col].to_numpy(dtype=float) * 1e-6  # Convert microseconds to seconds
    t -= t[0]

    dt = np.median(np.diff(t))
    fs = 1.0 / dt

    data = {"t": t, "dt": dt, "fs": fs}

    for i, name in enumerate(AXIS_NAMES):
        sp_col = f"setpoint[{i}]"
        gy_col = f"gyroADC[{i}]"
        if gy_col not in df.columns:
            gy_col = f"gyroData[{i}]"  # fallback name in some FW versions
        if sp_col not in df.columns:
            raise KeyError(
                f"Couldn't find '{sp_col}' in the CSV. Set "
                f"'debug_mode = SETPOINT' or check that your FW logs "
                f"setpoint directly. Available columns: {list(df.columns)[:20]}..."
            )
        data[f"setpoint_{name}"] = df[sp_col].to_numpy(dtype=float)
        data[f"gyro_{name}"] = df[gy_col].to_numpy(dtype=float)

    return data

def wiener_deconv_window(u, y, noise_reg):
    '''
    Single-window frequency-domain deconvolution:
        H(f) = (U*(f) Y(f)) / (|U(f)|^2 + noise_reg)
    Returns the impulse response h (same length as the window), time-shifted so h[0] corresponds to zero lag.
    '''
    n = len(u)
    win = np.hanning(n)
    U = np.fft.rfft(u * win)
    Y = np.fft.rfft(y * win)
    H = (np.conj(U) * Y) / (np.abs(U) ** 2 + noise_reg)
    h = np.fft.irfft(H, n)
    return h

def compute_axis_impulse_response(
    setpoint,
    gyro,
    fs,
    win_len_s=1.5,
    overlap=0.5,
    resp_len_s=0.5,
    activity_percentile=60,
    noise_reg=None,
):
    '''
    Slide windows across the log, deconvolve each, keep the
    high-stick-activity windows, average them (weighted by input
    energy) into one impulse response.

    win_len_s  -> window length in seconds (needs to be several x
                  longer than resp_len_s for the FFT deconv to be
                  well-posed)
    resp_len_s -> how much of the impulse response to keep (seconds)
    overlap    -> fraction of window overlap (0-0.9)
    activity_percentile : skip windows with setpoint activity below
                  this percentile (low-activity windows are mostly
                  noise and bias the estimate)
    noise_reg  -> Wiener regularization; if None, auto-set from
                  signal energy
    '''
    n_win = int(win_len_s * fs)
    step = int(n_win * (1 - overlap))
    n_resp = int(resp_len_s * fs)

    if noise_reg is None:
        noise_reg = 0.01 * np.mean(setpoint**2) * n_win

    windows_h = []
    weights = []

    for start in range(0, len(setpoint) - n_win, step):
        u = setpoint[start : start + n_win]
        y = gyro[start : start + n_win]

        activity = np.sum(u**2)
        if activity < 1e-6:
            continue

        h_full = wiener_deconv_window(u - u.mean(), y - y.mean(), noise_reg)
        h = h_full[:n_resp]

        windows_h.append(h)
        weights.append(activity)

    windows_h = np.array(windows_h)
    weights = np.array(weights)

    # keep only the more "active" windows (better SNR for identification)
    thresh = np.percentile(weights, activity_percentile)
    keep = weights >= thresh
    if keep.sum() < 5:
        keep = np.ones_like(weights, dtype=bool)  # fallback if too few

    h_avg = np.average(windows_h[keep], axis=0, weights=weights[keep])

    # Normalize so step response settles near 1.0 (typical convention
    # for "tracking" plots -- output/input ratio at steady state)
    step_resp = np.cumsum(h_avg)
    settle = np.mean(step_resp[-max(1, n_resp // 10) :])
    if abs(settle) > 1e-9:
        h_avg = h_avg / settle
        step_resp = step_resp / settle

    return h_avg, step_resp

# Compute impulse and step responses, plot figures
def get_responses(csv_path):
    '''
    Take blackbox log exported as a .csv fiLE
    Compute an impulse and step response, save to .png figure with metrics as footer
    Also export metrics as a text file with same name
    '''
    data = load_blackbox_csv(csv_path)
    fs = data["fs"]
    resp_len_s = 0.3
    t_resp = np.arange(int(resp_len_s * fs)) / fs

    # Stuff fr output locations
    out_dir = Path('Output metrics') # Output directory
    out_dir.mkdir(parents=True, exist_ok=True) # Create output directory if it does't exist

    base_name = Path(csv_path).name.split('.')[0] # Base name of file being read (remove file extension)
    out_path = out_dir / f'{base_name} Impulse and step responses.png'
    log_path = out_dir / f'{base_name} Impulse and step responses.txt'

    # --- capture prints while still showing them live in the cell ---
    buf = io.StringIO()
    class Tee(io.TextIOBase):
        def write(self, s):
            sys.__stdout__.write(s)   # live print, unchanged behavior
            buf.write(s)              # also buffer it
            return len(s)

    with contextlib.redirect_stdout(Tee()):
        print(f"Sample rate ~ {fs:.1f} Hz, duration {data['t'][-1]:.1f} s")

        fig, axes = plt.subplots(2, 3, figsize=(14, 7.5), sharex=True)

        for i, name in enumerate(AXIS_NAMES):
            setpoint = data[f"setpoint_{name}"]
            gyro = data[f"gyro_{name}"]
            h, step_resp = compute_axis_impulse_response(
                setpoint, gyro, fs, resp_len_s=resp_len_s
            )
            axes[0, i].plot(t_resp, h)
            axes[0, i].set_title(f"{name.capitalize()} Impulse Response")
            axes[0, i].axhline(0, color="gray", lw=0.5)
            axes[1, i].plot(t_resp, step_resp)
            axes[1, i].axhline(1.0, color="gray", linestyle="--", lw=0.8)
            axes[1, i].set_title(f"{name.capitalize()} Step Response")
            axes[1, i].set_xlabel("Time (s)")

            rise_idx = np.argmax(step_resp >= 0.9)
            overshoot = (np.max(step_resp) - 1.0) * 100
            print(
                f"{name:>6}: rise-to-90% = {t_resp[rise_idx]*1000:.1f} ms, "
                f"overshoot = {overshoot:.1f}%"
            )

        axes[0, 0].set_ylabel("Impulse response")
        axes[1, 0].set_ylabel("Step response")

        fig.suptitle(
            f"{base_name} — Impulse & Step Responses",
            fontsize=14, fontweight="bold", y=0.985
        )
        plt.tight_layout(rect=[0, 0.10, 1, 0.95])

        # embed captured metrics as a footer on the figure (Option B)
        metrics_text = buf.getvalue().strip()
        fig.text(
            0.02, 0.02, metrics_text,
            ha='left', va='bottom',
            fontsize=8, family='monospace',
            bbox=dict(facecolor='whitesmoke', edgecolor='lightgray', boxstyle='round,pad=0.5')
        )

        plt.savefig(out_path, dpi=150)
        #print(f"Saved plot to {out_path}")

    # write the log file after exiting the redirect block
    with open(log_path, 'w') as f:
        f.write(buf.getvalue())

    plt.show()
    plt.close() # Close figure to avoid slowdown if running computation on a lot of files
    # read back the log and print it in the cell 
    with open(log_path, 'r') as f:
        print(f.read())

In [ ]:
# Compute responses for a single file
get_responses(fn[0])

In [ ]:
# Compute responses for all files
for i in range (0, len(fn)):
    get_responses(fn[i])